# Gradient Attribution on Plan-Latents-Only Patching

Generate travel plans, capture residual stream activations during generation,
patch them into a re-run, and use gradient attribution to find which
layer/position latents most influence the continuation.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
print(f"Loaded {MODEL_ID} with {NUM_LAYERS} layers")

## Step 1: Generate Plans & Capture Activations

In [ ]:
TRAVEL_PROMPTS = {
    "Berlin": "My friends and I are going to the Berlin, Germany and go surfing. First name the universal steps to plan any international trip (short list of short phrases), then use the steps to book the trip.",
    "Bali": "I want to go to Bali, Indonesia and go swimming. First name the universal steps to plan any international trip (short list of short phrases), then use the steps to book the trip.",
    "Fiji": "I want to travel to Fiji and go skiing. First name the universal steps to plan any international trip (short list of short phrases), then use the steps to book the trip.",
}

PLAN_DELIMITER = "\n##"
ALT_DELIMITER = "\n**"


def format_chat(user_msg):
    """Format as Llama-3.1 chat template."""
    messages = [{"role": "user", "content": user_msg}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_plan_text(generated_text, prompt_text):
    """Extract plan section from generated text (up to second delimiter)."""
    gen_only = generated_text[len(prompt_text):]
    delim = PLAN_DELIMITER
    first = gen_only.find(delim)
    if first == -1:
        delim = ALT_DELIMITER
        first = gen_only.find(delim)
    if first == -1:
        print("WARNING: No delimiter found, using first 500 chars")
        return gen_only[:500]
    second = gen_only.find(delim, first + len(delim))
    if second == -1:
        print("WARNING: Second delimiter not found, using text up to end")
        return gen_only
    return gen_only[:second]


def generate_and_capture(prompt_text, max_new_tokens=300):
    """Generate plan and capture residual stream activations at generated positions.
    
    Returns:
        full_text: The complete generated text
        plan_text: Extracted plan section
        gen_activations: dict[layer_idx] -> tensor [1, gen_len, hidden_dim]
        prompt_len: number of prompt tokens
    """
    input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids.to(DEVICE)
    prompt_len = input_ids.shape[1]
    
    # Storage for activations captured during generation
    captured = {i: [] for i in range(NUM_LAYERS)}
    hooks = []
    
    def make_hook(layer_idx):
        def hook_fn(module, input, output):
            # output is a tuple; output[0] is the hidden state [batch, seq, hidden]
            # During generation, seq_len=1 for each new token (KV cache)
            hidden = output[0].detach().clone()
            captured[layer_idx].append(hidden)
        return hook_fn
    
    for i in range(NUM_LAYERS):
        h = model.model.layers[i].register_forward_hook(make_hook(i))
        hooks.append(h)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    for h in hooks:
        h.remove()
    
    # Concatenate captured activations: first call has full seq, rest have seq_len=1
    # We want only the generated token activations (positions prompt_len onward)
    gen_activations = {}
    for layer_idx in range(NUM_LAYERS):
        parts = captured[layer_idx]
        # First forward: [1, prompt_len, hidden] — take only generated part
        # But with KV cache, first call processes full prompt, subsequent calls process 1 token
        # parts[0] shape: [1, prompt_len, hidden] (prefill)
        # parts[1:] shape: [1, 1, hidden] each (generation steps)
        gen_parts = [p[:, -1:, :] for p in parts[1:]]  # skip prefill, take last token from each
        if len(gen_parts) > 0:
            gen_activations[layer_idx] = torch.cat(gen_parts, dim=1)  # [1, gen_len, hidden]
        else:
            gen_activations[layer_idx] = parts[0][:, prompt_len:, :]  # fallback: no KV cache
    
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    plan_text = extract_plan_text(full_text, tokenizer.decode(input_ids[0], skip_special_tokens=True))
    
    gen_len = gen_activations[0].shape[1]
    print(f"  Generated {gen_len} tokens, captured activations shape: {gen_activations[0].shape}")
    print(f"  Plan text ({len(plan_text)} chars): {plan_text[:150]}...")
    
    return full_text, plan_text, gen_activations, prompt_len

In [ ]:
# Generate plans and capture activations for each destination
results = {}

for dest, prompt in TRAVEL_PROMPTS.items():
    print(f"\n{'='*60}")
    print(f"Generating plan for: {dest}")
    print(f"{'='*60}")
    chat_prompt = format_chat(prompt)
    full_text, plan_text, gen_acts, prompt_len = generate_and_capture(chat_prompt)
    results[dest] = {
        "prompt": prompt,
        "chat_prompt": chat_prompt,
        "full_text": full_text,
        "plan_text": plan_text,
        "gen_activations": gen_acts,
        "prompt_len": prompt_len,
    }
    print(f"  Stored activations for {len(gen_acts)} layers")

## Step 2: Patched Forward Pass

In [ ]:
def build_patching_input(prompt, plan_text, dest):
    """Build conversation input: user prompt -> assistant plan -> user follow-up.
    
    Returns:
        input_ids: tokenized input
        plan_start: token index where plan starts
        plan_end: token index where plan ends
    """
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": plan_text},
        {"role": "user", "content": f"Now execute this plan specifically for {dest}."},
    ]
    full_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(full_chat, return_tensors="pt").input_ids.to(DEVICE)
    
    # Find plan token positions by tokenizing prefix up to plan, and prefix+plan
    pre_plan = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,  # assistant turn starts here
    )
    pre_plan_ids = tokenizer(pre_plan, return_tensors="pt").input_ids
    plan_start = pre_plan_ids.shape[1]
    
    pre_followup = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": plan_text}],
        tokenize=False,
    )
    pre_followup_ids = tokenizer(pre_followup, return_tensors="pt").input_ids
    plan_end = pre_followup_ids.shape[1]
    
    plan_token_len = plan_end - plan_start
    print(f"  Input length: {input_ids.shape[1]}, plan tokens: [{plan_start}:{plan_end}] ({plan_token_len} tokens)")
    
    return input_ids, plan_start, plan_end


def patched_forward(input_ids, plan_start, plan_end, gen_activations):
    """Forward pass with residual stream patched at plan positions.
    
    Returns logits at last position.
    """
    plan_len = plan_end - plan_start
    hooks = []
    
    def make_patch_hook(layer_idx):
        def hook_fn(module, input, output):
            hidden = output[0]  # [batch, seq, hidden]
            act = gen_activations[layer_idx]  # [1, gen_len, hidden]
            # Patch plan positions with saved generation activations
            patch_len = min(plan_len, act.shape[1])
            hidden = hidden.clone()
            hidden[:, plan_start:plan_start + patch_len, :] = act[:, :patch_len, :].to(hidden.device)
            return (hidden,) + output[1:]
        return hook_fn
    
    for i in range(NUM_LAYERS):
        h = model.model.layers[i].register_forward_hook(make_patch_hook(i))
        hooks.append(h)
    
    with torch.no_grad():
        out = model(input_ids)
    
    for h in hooks:
        h.remove()
    
    return out.logits[:, -1, :]  # [1, vocab_size]

In [ ]:
# Run patched forward for each destination
for dest, r in results.items():
    print(f"\n--- {dest} ---")
    input_ids, plan_start, plan_end = build_patching_input(r["prompt"], r["plan_text"], dest)
    patched_logits = patched_forward(input_ids, plan_start, plan_end, r["gen_activations"])
    r["input_ids"] = input_ids
    r["plan_start"] = plan_start
    r["plan_end"] = plan_end
    r["patched_logits"] = patched_logits
    top_token = patched_logits[0].argmax().item()
    print(f"  Patched top token: {tokenizer.decode([top_token])!r} (id={top_token})")

## Step 3: Unpatched Baseline & Metric

In [ ]:
for dest, r in results.items():
    print(f"\n--- {dest} ---")
    with torch.no_grad():
        out = model(r["input_ids"])
    unpatched_logits = out.logits[:, -1, :]  # [1, vocab_size]
    r["unpatched_logits"] = unpatched_logits
    
    # Metric: logit diff at patched model's top token
    top_token = r["patched_logits"][0].argmax().item()
    logit_diff = (r["patched_logits"][0, top_token] - unpatched_logits[0, top_token]).item()
    r["logit_diff"] = logit_diff
    r["top_token"] = top_token
    
    unpatched_top = unpatched_logits[0].argmax().item()
    print(f"  Patched top: {tokenizer.decode([top_token])!r}, Unpatched top: {tokenizer.decode([unpatched_top])!r}")
    print(f"  Logit diff: {logit_diff:.4f}")

## Step 4: Gradient Attribution

In [ ]:
def gradient_attribution(input_ids, plan_start, plan_end, gen_activations, top_token):
    """Compute gradient attribution for patched plan-token activations.
    
    Attribution = grad * (patched_activation - clean_activation) per layer/position.
    
    Returns:
        attribution: tensor [NUM_LAYERS, plan_len] (summed over hidden dim)
        plan_tokens: list of decoded plan token strings
    """
    plan_len = plan_end - plan_start
    
    # First: get clean activations at plan positions (no patching)
    clean_acts = {}
    clean_hooks = []
    
    def make_capture_hook(layer_idx):
        def hook_fn(module, input, output):
            clean_acts[layer_idx] = output[0][:, plan_start:plan_end, :].detach().clone()
        return hook_fn
    
    for i in range(NUM_LAYERS):
        h = model.model.layers[i].register_forward_hook(make_capture_hook(i))
        clean_hooks.append(h)
    
    with torch.no_grad():
        model(input_ids)
    
    for h in clean_hooks:
        h.remove()
    
    # Create leaf tensors for patched activations with grad enabled
    patch_tensors = {}
    for layer_idx in range(NUM_LAYERS):
        act = gen_activations[layer_idx][:, :plan_len, :].to(dtype=torch.float32, device=DEVICE)
        patch_tensors[layer_idx] = act.clone().detach().requires_grad_(True)
    
    # Patched forward with gradient tracking
    patch_hooks = []
    
    def make_grad_patch_hook(layer_idx):
        def hook_fn(module, input, output):
            hidden = output[0].clone()
            # Cast to float32 for gradient computation at the patch site
            hidden_patch = hidden[:, plan_start:plan_end, :].to(torch.float32)
            # Replace with our grad-enabled tensor
            patched = patch_tensors[layer_idx]
            # Blend back: replace plan positions
            hidden = hidden.to(torch.float32)
            hidden[:, plan_start:plan_end, :] = patched
            return (hidden.to(output[0].dtype),) + output[1:]
        return hook_fn
    
    for i in range(NUM_LAYERS):
        h = model.model.layers[i].register_forward_hook(make_grad_patch_hook(i))
        patch_hooks.append(h)
    
    # Forward with gradients enabled for the model
    out = model(input_ids)
    logits = out.logits[0, -1, :].float()  # [vocab_size]
    
    for h in patch_hooks:
        h.remove()
    
    # Compute metric and backward
    metric = logits[top_token]
    metric.backward()
    
    # Collect attributions
    attribution = torch.zeros(NUM_LAYERS, plan_len)
    for layer_idx in range(NUM_LAYERS):
        grad = patch_tensors[layer_idx].grad  # [1, plan_len, hidden]
        if grad is None:
            print(f"  WARNING: grad is None for layer {layer_idx}")
            continue
        clean = clean_acts[layer_idx][:, :plan_len, :].to(torch.float32)
        patched = patch_tensors[layer_idx].detach()
        # Attribution = grad * (patched - clean), summed over hidden dim
        attr = (grad * (patched - clean)).sum(dim=-1).sum(dim=0)  # [plan_len]
        attribution[layer_idx] = attr.cpu()
    
    # Decode plan tokens for labels
    plan_token_ids = input_ids[0, plan_start:plan_end].tolist()
    plan_tokens = [tokenizer.decode([tid]) for tid in plan_token_ids]
    
    model.zero_grad()
    
    return attribution, plan_tokens

In [ ]:
# Run gradient attribution for each destination
for dest, r in results.items():
    print(f"\n{'='*60}")
    print(f"Gradient attribution for: {dest}")
    print(f"{'='*60}")
    attribution, plan_tokens = gradient_attribution(
        r["input_ids"], r["plan_start"], r["plan_end"],
        r["gen_activations"], r["top_token"],
    )
    r["attribution"] = attribution
    r["plan_tokens"] = plan_tokens
    print(f"  Attribution shape: {attribution.shape}")
    print(f"  Attribution range: [{attribution.min():.4f}, {attribution.max():.4f}]")
    print(f"  Non-zero layers: {(attribution.abs().sum(dim=1) > 0).sum().item()}/{NUM_LAYERS}")

## Step 5: Visualization

In [ ]:
def plot_attribution_heatmap(attribution, plan_tokens, title, ax=None, max_tokens=80):
    """Plot layer x position attribution heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(12, len(plan_tokens) * 0.3), 8))
    
    data = attribution[:, :max_tokens].numpy()
    tokens = plan_tokens[:max_tokens]
    
    # Symmetric color scale
    vmax = max(abs(data.min()), abs(data.max()))
    if vmax == 0:
        vmax = 1.0
    
    im = ax.imshow(data, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   interpolation="nearest")
    ax.set_xlabel("Plan Token Position")
    ax.set_ylabel("Layer")
    ax.set_title(title)
    ax.set_yticks(range(0, data.shape[0], 4))
    
    # Token labels on x-axis (show every Nth to avoid overlap)
    n_tokens = len(tokens)
    step = max(1, n_tokens // 30)
    tick_positions = list(range(0, n_tokens, step))
    tick_labels = [tokens[i].replace('\n', '\\n') for i in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
    
    plt.colorbar(im, ax=ax, label="Attribution")
    return ax

In [ ]:
# Individual heatmaps per destination
for dest, r in results.items():
    fig, ax = plt.subplots(figsize=(max(14, len(r['plan_tokens']) * 0.3), 8))
    plot_attribution_heatmap(
        r["attribution"], r["plan_tokens"],
        f"Gradient Attribution: {dest}\nLogit diff = {r['logit_diff']:.4f}, "
        f"top token = {tokenizer.decode([r['top_token']])!r}",
        ax=ax,
    )
    plt.tight_layout()
    plt.savefig(f"results/attribution_{dest.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Cross-destination comparison (side by side)
dests = list(results.keys())
fig, axes = plt.subplots(1, len(dests), figsize=(8 * len(dests), 8))
if len(dests) == 1:
    axes = [axes]

for ax, dest in zip(axes, dests):
    r = results[dest]
    plot_attribution_heatmap(
        r["attribution"], r["plan_tokens"],
        f"{dest} (logit diff={r['logit_diff']:.2f})",
        ax=ax,
    )

plt.suptitle("Cross-Destination Attribution Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("results/attribution_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Layer-aggregated attribution (sum over positions per layer)
fig, ax = plt.subplots(figsize=(10, 5))
for dest, r in results.items():
    layer_attr = r["attribution"].abs().sum(dim=1).numpy()  # [NUM_LAYERS]
    ax.plot(range(NUM_LAYERS), layer_attr, marker="o", markersize=3, label=dest)

ax.set_xlabel("Layer")
ax.set_ylabel("|Attribution| (summed over positions)")
ax.set_title("Layer-wise Attribution Magnitude")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/attribution_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()